In [53]:
import pandas as pd
from dataclasses import dataclass
from dateutil import parser as dateparser
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel, field_validator
from typing import Optional

In [55]:
df = pd.read_csv("T_ONTIME_REPORTING.csv")
df

,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_TIME,ARR_DELAY
0,2025,1,1,3,10140,1014005,30140,ABQ,10397,1039707,30397,ATL,635,625.0,-10.0,1132,1110.0,-22.0
1,2025,1,1,3,10140,1014005,30140,ABQ,10397,1039707,30397,ATL,1135,1144.0,9.0,1633,1618.0,-15.0
2,2025,1,1,3,10208,1020803,30208,AGS,11057,1105703,31057,CLT,705,653.0,-12.0,820,802.0,-18.0
3,2025,1,1,3,10208,1020803,30208,AGS,11057,1105703,31057,CLT,1058,1056.0,-2.0,1208,1150.0,-18.0
4,2025,1,1,3,10208,1020803,30208,AGS,11057,1105703,31057,CLT,1830,1820.0,-10.0,1940,1943.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50211,2025,1,31,5,15624,1562404,31504,VPS,10397,1039707,30397,ATL,922,932.0,10.0,1135,1137.0,2.0
50212,2025,1,31,5,15624,1562404,31504,VPS,10397,1039707,30397,ATL,1159,1154.0,-5.0,1413,1408.0,-5.0
50213,2025,1,31,5,15624,1562404,31504,VPS,10397,1039707,30397,ATL,1455,1453.0,-2.0,1710,1705.0,-5.0
50214,2025,1,31,5,15624,1562404,31504,VPS,10397,1039707,30397,ATL,1700,1736.0,36.0,1920,1945.0,25.0


In [107]:
df.columns = [c.upper() for c in df.columns]

needed = {"ORIGIN", "DEST", "DEP_DELAY"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

time_col = "CRS_DEP_TIME" if "CRS_DEP_TIME" in df.columns else "DEP_TIME" if "DEP_TIME" in df.columns else None
if time_col is None:
    raise ValueError("Need CRS_DEP_TIME or DEP_TIME")
if "ORIGIN" not in df.columns or "DEST" not in df.columns or "DEP_DELAY" not in df.columns:
    raise ValueError("Need ORIGIN, DEST, DEP_DELAY")

In [111]:
def hhmm_to_hour(x):
    try:
        x = str(int(float(x))).zfill(3)
        hh = int(x[:-2]) if len(x) > 2 else 0
        return hh if 0 <= hh <= 23 else None
    except Exception:
        return None

df["DEP_HOUR"] = df[time_col].apply(hhmm_to_hour)
df = df.dropna(subset=["DEP_HOUR", "DEP_DELAY"]).copy() 
df["DEP_HOUR"] = df["DEP_HOUR"].astype(int)
if df.empty:
    raise ValueError("No usable rows after cleaning; check your CSV.")

ROUTE_HOUR_MEANS = df.groupby(["ORIGIN", "DEST", "DEP_HOUR"])["DEP_DELAY"].mean()
ROUTE_MEANS = df.groupby(["ORIGIN", "DEST"])["DEP_DELAY"].mean()
GLOBAL_MEAN = float(df["DEP_DELAY"].mean())

In [157]:
class DelayModel:
    def __init__(self, rh_means, r_means, g_mean):
        self.rh_means = rh_means
        self.r_means = r_means
        self.g_mean = g_mean
        self.version = "notebook_v3"

    def predict(self, dep_airport: Optional[str], arr_airport: str, dep_local_iso: str, arr_local_iso: str):
        try:
            dep_hour = dateparser.isoparse(dep_local_iso).hour
        except Exception as e:
            raise ValueError(f"Invalid departure_time_local: {e}")
        details = {"hour": dep_hour, "used": None}

        if dep_airport:
            k = (dep_airport, arr_airport, dep_hour)
            if k in self.rh_means.index:
                details["used"] = "route_hour_mean"
                return float(self.rh_means.loc[k]), details
            rk = (dep_airport, arr_airport)
            if rk in self.r_means.index:
                details["used"] = "route_mean"
                return float(self.r_means.loc[rk]), details

        details["used"] = "global_mean"
        return float(self.g_mean), details

In [115]:
from fastapi import FastAPI, Query, HTTPException
from pydantic import BaseModel, field_validator
from typing import Optional

In [159]:
app = FastAPI(title="Airline Delay Prediction API (Notebook)", version="1.0.0")
app.state.model = DelayModel(ROUTE_HOUR_MEANS, ROUTE_MEANS, GLOBAL_MEAN)

In [161]:
def norm_airport(v: Optional[str]) -> Optional[str]:
    if v is None:
        return None
    v = v.strip().upper()
    if len(v) != 3 or not v.isalpha():
        raise ValueError("Airport codes must be 3 letters, e.g., LAX, JFK, ATL.")
    return v

In [163]:
class PredictResponse(BaseModel):
    average_departure_delay_min: float
    route: Optional[str] = None
    details: dict
    model_version: str = "notebook_v1"

In [123]:
class PredictParams(BaseModel):
    arrival_airport: str
    departure_time_local: str
    arrival_time_local: str
    departure_airport: Optional[str] = None

    @field_validator("arrival_airport")
    @classmethod
    def _valid_arr(cls, v):
        return norm_airport(v)

    @field_validator("departure_airport")
    @classmethod
    def _valid_dep(cls, v):
        return norm_airport(v) if v is not None else v


In [125]:
@app.get("/")
def root():
    return {"status": "ok", "message": "API running (notebook mode)", "version": app.version}


In [181]:
@app.get("/predict/delays", response_model=PredictResponse)
def predict_delays(
    arrival_airport: str = Query(..., description="IATA, e.g., LAX"),
    departure_time_local: str = Query(..., description="ISO 8601, e.g., 2025-10-26T09:00"),
    arrival_time_local: str = Query(..., description="ISO 8601, e.g., 2025-10-26T12:00"),
    departure_airport: Optional[str] = Query(None, description="IATA, e.g., JFK"),
):
    # validate first
    params = PredictParams(
        arrival_airport=arrival_airport,
        departure_time_local=departure_time_local,
        arrival_time_local=arrival_time_local,
        departure_airport=departure_airport,
    )

    try:
        avg, details = app.state.model.predict(
            dep_airport=params.departure_airport,
            arr_airport=params.arrival_airport,
            dep_local_iso=params.departure_time_local,
            arr_local_iso=params.arrival_time_local,
        )
    except ValueError as e:
        # bad timestamp, etc.
        raise HTTPException(status_code=422, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {type(e).__name__}: {e}")

    route = f"{params.departure_airport}->{params.arrival_airport}" if params.departure_airport else None
    return {
        "average_departure_delay_min": round(float(avg), 2),
        "route": route,
        "details": details,
        "model_version": app.state.model.version,
    }

In [183]:
from fastapi.testclient import TestClient
client = TestClient(app)

In [185]:
r = client.get("/")
print("Test 1: / ->", r.status_code, r.json())
assert r.status_code == 200
assert "status" in r.json()
assert "version" in r.json()

Test 1: / -> 200 {'status': 'ok', 'message': 'API running (notebook mode)', 'version': '1.0.0'}


In [187]:
params = {
    "arrival_airport": "ORD",
    "departure_airport": "ATL",
    "departure_time_local": "2025-10-26T14:00",
    "arrival_time_local": "2025-10-26T16:00",
}
r = client.get("/predict/delays", params=params)
print("\nTest 2: Valid request ->", r.status_code, r.json())


Test 2: Valid request -> 200 {'average_departure_delay_min': 7.05, 'route': 'ATL->ORD', 'details': {'hour': 14, 'used': 'route_hour_mean'}, 'model_version': 'notebook_v3'}


In [189]:
params = {
    "arrival_airport": "LAX",
    "arrival_time_local": "2025-10-26T12:00",
}
r = client.get("/predict/delays", params=params)
print("\nTest 3: Missing param →", r.status_code, r.json())
assert r.status_code == 422


Test 3: Missing param → 422 {'detail': [{'type': 'missing', 'loc': ['query', 'departure_time_local'], 'msg': 'Field required', 'input': None}]}


In [191]:
params = {"arrival_airport": "LA", "departure_time_local": "2025-10-26T09:00", "arrival_time_local": "2025-10-26T12:00"}
r = client.get("/predict/delays", params=params)
print("\nTest 4:", r.status_code, r.json())
assert r.status_code == 422

ValidationError: 1 validation error for PredictParams
arrival_airport
  Value error, Airport codes must be 3 letters, e.g., LAX, JFK, ATL. [type=value_error, input_value='LA', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [193]:
params = {
    "arrival_airport": "LAX",
    "departure_airport": "JFK",
    "departure_time_local": "not-a-timestamp",
    "arrival_time_local": "2025-10-26T12:00",
}
r = client.get("/predict/delays", params=params)
print("\nTest 5:", r.status_code, r.json())
assert r.status_code == 422


Test 5: 422 {'detail': "Invalid departure_time_local: invalid literal for int() with base 10: b'not-'"}


In [195]:
params = {
    "arrival_airport": "ANC",
    "departure_airport": "HNL",
    "departure_time_local": "2025-10-26T05:00",
    "arrival_time_local": "2025-10-26T12:00",
}
r = client.get("/predict/delays", params=params)
print("\nTest 6:", r.status_code, r.json())
assert r.status_code == 200
assert r.json()["details"]["used"] in {"route_hour_mean", "route_mean", "global_mean"}

print("\n All Part C tests passed successfully!")


Test 6: 200 {'average_departure_delay_min': 15.36, 'route': 'HNL->ANC', 'details': {'hour': 5, 'used': 'global_mean'}, 'model_version': 'notebook_v3'}

 All Part C tests passed successfully!
